In [1]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig
from peft.tuners.lora.config import CordaConfig
from peft.tuners.lora.corda import preprocess_corda
from trl import SFTTrainer, SFTConfig


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

/home/omar-elmasaoudi/miniconda3/envs/ml_dev_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda



### CorDA Module

Given our task of enabling Qwen models to perform better on specific downstream tasks, a common issue with QLoRA/LoRA and other PEFT methods is that these modules often forget the context of the downstream task and may also overwrite pre-trained world knowledge (stored in the parameters of the base model).
To address this, we add an additional module to our fine-tuning pipeline called CorDA.



### What is CorDA

CorDA, short for Context-Oriented Decomposition Adaptation, decomposes each linear layer’s weight matrix into subspaces that reflect the contextual importance of features for a given task.
This allows us to separate task-specific components (directions strongly activated for the target task) from generalizable components (directions associated with pre-trained knowledge).

We obtain these subspaces through the following process:

1. **Collect activations:**
   For a specified task (e.g., QA or another evaluation dataset), pass representative samples $(x, \hat{y})$ through the model and record the pre-activation hidden outputs from each linear layer:
   $$
   X \in \mathbb{R}^{N \times D}
   $$
   where $N$ is the number of tokens in the batch and $D$ is the hidden dimension.

2. **Compute the feature covariance matrix:**
   $$
   \Sigma_X = X^T X
   $$
   This covariance matrix captures how different neurons (features) co-activate across samples, revealing which parts of the representation space are most relevant to the task context.

3. **Compute the context-oriented weight matrix:**
   $$
   A = W \Sigma_X
   $$
   where $W \in \mathbb{R}^{D_{\text{out}} \times D_{\text{in}}}$.
   This step reorients the layer’s weights toward the input directions that are most active in the dataset’s context.

4. **Perform Singular Value Decomposition (SVD):**
   Decompose $A$ as:
   $$
   A = U S V^T
   $$
   where the diagonal matrix $S$ contains singular values $\sigma_1, \sigma_2, \dots$ that rank the importance of each component.

   * Large singular values capture dominant, task-relevant directions.
   * Smaller singular values capture more general, residual directions.

   The top-$r$ components (largest singular values) represent highly task-specific subspaces, while the smallest-$r$ components correspond to more generalizable pathways.

5. **Selective adaptation:**
   CorDA allows us to choose which subspace to make trainable:

   * *Task-specialized mode:* Freeze the small-$\sigma$ components and fine-tune the large-$\sigma$ ones.
     → Maximizes performance on the downstream task but may cause catastrophic forgetting.
   * *Knowledge-preserved mode:* Freeze the large-$\sigma$ components and fine-tune the small-$\sigma$ ones.
     → Retains general world knowledge while improving task performance with minimal forgetting.




### Init model 

In [2]:

model_name = "Qwen/Qwen2.5-0.5B"

model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token_id = tokenizer.eos_token_id


`torch_dtype` is deprecated! Use `dtype` instead!


### Load dummy dataset

In [3]:

sampled_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:8]")
dataset = load_dataset("imdb", split="train[:8]")

### Define LoRa & CorDA configs

In [4]:

corda_config = CordaConfig(
    corda_method="kpm",
    use_float16_for_covariance=True
)

lora_config_corda_init = LoraConfig(
    init_lora_weights="corda",
    corda_config=corda_config,
    target_modules=["self_attn.q_proj", "self_attn.v_proj"],
    task_type="CAUSAL_LM"
)

### Define Model Run loop

In [5]:
def run_model():
    model_device = next(model_fp16.parameters()).device
    for example in sampled_dataset:
        text = example["text"]
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=128,
        )
        if inputs["input_ids"].numel() == 0:
            continue
        inputs = {k: v.to(model_device) for k, v in inputs.items()}
        with torch.no_grad():
            _ = model_fp16(**inputs)

### Run CorDA on fp16 model and save to disk

In [6]:

preprocess_corda(model_fp16, lora_config_corda_init, run_model=run_model)


peft_model_fp16 = get_peft_model(model_fp16, lora_config_corda_init)


peft_model_fp16.print_trainable_parameters()

adapter_save_path = "../model/corda_qwen_adapter"
peft_model_fp16.save_pretrained(adapter_save_path)
print("saved adapter with CorDA init:", adapter_save_path)


trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093
saved adapter with CorDA init: ../model/corda_qwen_adapter


### Define 4bit Quantized configs for qwen model

In [7]:
from peft import PeftModel, PeftConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model_4bit = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

lora_config_runtime = LoraConfig(
    r=lora_config_corda_init.r,
    lora_alpha=lora_config_corda_init.lora_alpha,
    lora_dropout=lora_config_corda_init.lora_dropout,
    target_modules=lora_config_corda_init.target_modules,
    task_type="CAUSAL_LM",
)


peft_model_quant = get_peft_model(model_4bit, lora_config_runtime)


### Load CorDA initialized adapter weights unto the model

In [8]:
peft_model_quant.load_adapter(adapter_save_path, adapter_name="default", is_trainable=True)

peft_model_quant.print_trainable_parameters()
print("quantized model now has CorDA-initialized LoRA weights loaded.")


trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093
quantized model now has CorDA-initialized LoRA weights loaded.


### Tune the model in a training loop

In [ ]:
from datetime import datetime 

run_date = datetime.datetime.now()

training_args = SFTConfig(
    dataset_text_field="text",
    max_length=128,
    num_train_epochs=100,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    output_dir="../model/corda_qwen_checkpoints",
    report_to=None,
)

trainer = SFTTrainer(
    model=peft_model_quant,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()
saved_model_path = f"../model/corda_qwen2.5_0.5B_quant_lora-{run_date}"
peft_model_quant.save_pretrained(saved_model_path)
print("training done and adapter saved.")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,3.454400
10,3.335100
15,3.223800
20,3.051300
25,2.948300
30,2.828100
35,2.718800
40,2.536900
45,2.427200
50,2.284200


training done and adapter saved.
